In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="kode_api")
project = rf.workspace("name").project("project")
version = project.version(3)
dataset = version.download("yolov8")




In [ ]:
!pip install ultralytics
!yolo task=detect mode=train epochs=100 data=/content/drive/MyDrive/672-Terbaru/data.yaml model=yolov8m.pt imgsz=640 batch=8 patience=128

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.7 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import cv2
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from ultralytics import YOLO
import cv2
import time

# input model
model = YOLO("/content/drive/best.pt")

# input video
cap = cv2.VideoCapture("/content/Video_uji.mp4")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter(
    "/content/output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    video_fps,
    (width, height)
)

# penyimpanan data evaluasi
fps_list = []
latency_list = []
inference_list = []

# warna tiap kelas
class_colors = {
    'igneous_diorite': (139, 0, 0),
    'sedimentary_coal': (200, 220, 255),
    'metamorphic_marble': (255, 200, 100)
}

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    # ===============================
    # Hitung waktu proses frame
    # ===============================
    start_time = time.time()

    results = model.track(
        frame,
        persist=True,
        tracker="botsort.yaml",
        conf=0.25,
        verbose=False
    )

    end_time = time.time()

    result = results[0]

    # ===============================
    # Evaluasi Kecepatan
    # ===============================
    latency = (end_time - start_time) * 1000  # ms

    if latency > 0:
        fps_real = 1000 / latency
    else:
        fps_real = 0

    inference_time = result.speed['inference']

    fps_list.append(fps_real)
    latency_list.append(latency)
    inference_list.append(inference_time)

    # ===============================
    # Bounding Box
    # ===============================
    if result.boxes.id is not None:

        boxes = result.boxes
        ids = boxes.id.cpu().numpy()
        classes = boxes.cls.cpu().numpy()

        for box, obj_id, cls in zip(boxes, ids, classes):

            cls_id = int(cls)
            class_name = model.names[cls_id].lower()

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            color = class_colors.get(class_name, (0, 255, 0))

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                color,
                3
            )

            label = f"{class_name} {conf:.2f}"

            if class_name in ["sedimentary_coal", "metamorphic_marble"]:
                font_color = (0, 0, 0)
            else:
                font_color = (255, 255, 255)

            (w, h), _ = cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                2
            )

            cv2.rectangle(
                frame,
                (x1, y1 - h - 10),
                (x1 + w, y1),
                color,
                -1
            )

            cv2.putText(
                frame,
                label,
                (x1, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                font_color,
                2
            )

    # ===============================
    # Total Deteksi
    # ===============================
    current_total = len(result.boxes)

    cv2.putText(
        frame,
        f"Total Terdeteksi: {current_total}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )

    # ===============================
    # FPS
    # ===============================
    cv2.putText(
        frame,
        f"FPS: {fps_real:.2f}",
        (10, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2
    )

    # ===============================
    # Inference Time
    # ===============================
    cv2.putText(
        frame,
        f"Inference: {inference_time:.2f} ms",
        (10, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2
    )

    # ===============================
    # Latency
    # ===============================
    cv2.putText(
        frame,
        f"Latency: {latency:.2f} ms",
        (10, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2
    )

    out.write(frame)

# ===============================
# Statistik Hasil Pengujian
# ===============================

fps_array = np.array(fps_list)
latency_array = np.array(latency_list)
inference_array = np.array(inference_list)

# ===============================
# Mean
# ===============================
avg_fps = np.mean(fps_array)
avg_latency = np.mean(latency_array)
avg_inference = np.mean(inference_array)

# ===============================
# Delta (Standar Deviasi)
# ===============================
delta_fps = np.std(fps_array, ddof=1)
delta_latency = np.std(latency_array, ddof=1)
delta_inference = np.std(inference_array, ddof=1)

# ===============================
# Ketelitian (Coefficient of Variation)
# ===============================
cv_fps = (delta_fps / avg_fps) * 100
cv_latency = (delta_latency / avg_latency) * 100
cv_inference = (delta_inference / avg_inference) * 100

print("\n========== HASIL PENGUJIAN ==========\n")

print(f"FPS")
print(f"Rata-rata      : {avg_fps:.2f}")
print(f"Delta (SD)     : ±{delta_fps:.2f}")
print(f"CV             : {cv_fps:.2f}%")

print()

print(f"Inference Time")
print(f"Rata-rata      : {avg_inference:.2f} ms")
print(f"Delta (SD)     : ±{delta_inference:.2f} ms")
print(f"CV             : {cv_inference:.2f}%")

print()

print(f"Latency")
print(f"Rata-rata      : {avg_latency:.2f} ms")
print(f"Delta (SD)     : ±{delta_latency:.2f} ms")
print(f"CV             : {cv_latency:.2f}%")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 199ms
Prepared 1 package in 31ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



NameError: name 'np' is not defined

In [ ]:
from ultralytics import YOLO
import cv2
import time
import numpy as np

# ==============================
# Load Model
# ==============================
model = YOLO("/content/drive/MyDrive/Interval_data_baru/runs_50_baru1/detect/train/weights/best.pt")

# ==============================
# Load Video
# ==============================
cap = cv2.VideoCapture("/content/drive/MyDrive/Interval_data_baru/Video_uji/multi3.mp4")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter(
    "/content/output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    video_fps,
    (width, height)
)

# ==============================
# Penyimpanan Data
# ==============================
fps_list = []
latency_list = []
inference_list = []

# ==============================
# Warna Bounding Box
# ==============================
class_colors = {
    'igneous_diorite': (139, 0, 0),
    'sedimentary_coal': (200, 220, 255),
    'metamorphic_marble': (255, 200, 100)
}

frame_count = 0

# ==============================
# Proses Video
# ==============================
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_count += 1

    # ---------------------------------------
    # Hitung waktu proses
    # ---------------------------------------
    start_time = time.perf_counter()

    results = model.track(
        frame,
        persist=True,
        tracker="botsort.yaml",
        conf=0.25,
        verbose=False
    )

    end_time = time.perf_counter()

    result = results[0]

    # ---------------------------------------
    # Evaluasi
    # ---------------------------------------
    latency = (end_time - start_time) * 1000

    fps_real = 1 / (end_time - start_time)

    inference_time = result.speed.get("inference", 0)

    fps_list.append(fps_real)
    latency_list.append(latency)
    inference_list.append(inference_time)

    # ---------------------------------------
    # Bounding Box
    # ---------------------------------------
    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes
        ids = boxes.id.cpu().numpy()
        classes = boxes.cls.cpu().numpy()

        for box, obj_id, cls in zip(boxes, ids, classes):

            cls_id = int(cls)
            class_name = model.names[cls_id].lower()

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            color = class_colors.get(class_name, (0,255,0))

            cv2.rectangle(frame,(x1,y1),(x2,y2),color,3)

            label = f"{class_name} {conf:.2f}"

            if class_name in ["sedimentary_coal","metamorphic_marble"]:
                font_color=(0,0,0)
            else:
                font_color=(255,255,255)

            (w,h),_ = cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                2
            )

            cv2.rectangle(frame,(x1,y1-h-10),(x1+w,y1),color,-1)

            cv2.putText(
                frame,
                label,
                (x1,y1-5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                font_color,
                2
            )

    # ---------------------------------------
    # Total Objek
    # ---------------------------------------
    total_obj = len(result.boxes)

    cv2.putText(
        frame,
        f"Total : {total_obj}",
        (10,30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0,0,255),
        2
    )

    # ---------------------------------------
    # FPS
    # ---------------------------------------
    cv2.putText(
        frame,
        f"FPS : {fps_real:.2f}",
        (10,60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,0,0),
        2
    )

    # ---------------------------------------
    # Inference
    # ---------------------------------------
    cv2.putText(
        frame,
        f"Inference : {inference_time:.2f} ms",
        (10,90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,0,0),
        2
    )

    # ---------------------------------------
    # Latency
    # ---------------------------------------
    cv2.putText(
        frame,
        f"Latency : {latency:.2f} ms",
        (10,120),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,0,0),
        2
    )

    out.write(frame)

# ==============================
# Release
# ==============================
cap.release()
out.release()
cv2.destroyAllWindows()

# ==============================
# Statistik
# ==============================
fps_array = np.array(fps_list)
latency_array = np.array(latency_list)
inference_array = np.array(inference_list)

# Mean
avg_fps = np.mean(fps_array)
avg_latency = np.mean(latency_array)
avg_inference = np.mean(inference_array)

# Delta (Standar Deviasi)
delta_fps = np.std(fps_array, ddof=1)
delta_latency = np.std(latency_array, ddof=1)
delta_inference = np.std(inference_array, ddof=1)

# Ketelitian (CV)
cv_fps = (delta_fps / avg_fps) * 100
cv_latency = (delta_latency / avg_latency) * 100
cv_inference = (delta_inference / avg_inference) * 100

# ==============================
# Output
# ==============================
print("\n==============================")
print("      HASIL PENGUJIAN")
print("==============================")

print(f"Jumlah Frame             : {frame_count}")

print("\nFPS")
print(f"Rata-rata                : {avg_fps:.2f} FPS")
print(f"Delta (SD)               : ±{delta_fps:.2f}")
print(f"Ketidaktelitian (CV)     : {cv_fps:.2f}%")

print("\nLATENCY")
print(f"Rata-rata                : {avg_latency:.2f} ms")
print(f"Delta (SD)               : ±{delta_latency:.2f}")
print(f"Ketidaktelitian (CV)     : {cv_latency:.2f}%")

print("\nINFERENCE TIME")
print(f"Rata-rata                : {avg_inference:.2f} ms")
print(f"Delta (SD)               : ±{delta_inference:.2f}")
print(f"Ketidaktelitian (CV)     : {cv_inference:.2f}%")

print("\nVideo hasil disimpan pada : /content/output.mp4")


      HASIL PENGUJIAN
Jumlah Frame             : 3021

FPS
Rata-rata                : 19.33 FPS
Delta (SD)               : ±3.31
Ketidaktelitian (CV)     : 17.10%

LATENCY
Rata-rata                : 53.57 ms
Delta (SD)               : ±12.31
Ketidaktelitian (CV)     : 22.98%

INFERENCE TIME
Rata-rata                : 21.56 ms
Delta (SD)               : ±2.53
Ketidaktelitian (CV)     : 11.73%

Video hasil disimpan pada : /content/output.mp4
